In [1]:
import itertools

"""
LSTM-Rec baseline for Retailrocket dataset.
Predicts the next item a user will interact with given their session history.

Dataset: https://www.kaggle.com/datasets/retailrocket/ecommerce-dataset
Expected file: events.csv (columns: timestamp, visitorid, event, itemid, transactionid)
"""

import numpy as np
import pandas as pd
from collections import Counter

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence, pad_packed_sequence

In [2]:
EVENT_WEIGHTS = {"view": 1.0, "addtocart": 5.0, "transaction": 10.0}
TIME_DECAY_HALF_LIFE_DAYS = 90  # weight halves every 30 days

df = pd.read_csv("events.csv")

# keep only known event types
df = df[df["event"].isin(EVENT_WEIGHTS.keys())].copy()
df["weight"] = df["event"].map(EVENT_WEIGHTS)
df.sort_values(["visitorid", "timestamp"], inplace=True)

# --- time decay: exponential decay based on recency ---
# timestamp is in milliseconds
df["ts_seconds"] = df["timestamp"] / 1000.0
max_ts = df["ts_seconds"].max()
df["days_ago"] = (max_ts - df["ts_seconds"]) / 86400.0
# decay_rate = np.log(2) / TIME_DECAY_HALF_LIFE_DAYS
# df["time_decay"] = np.exp(-decay_rate * df["days_ago"])

# combined weight = event_weight * time_decay
# df["weight"] = df["weight"] * df["time_decay"]

w_min = df["weight"].min()
w_max = df["weight"].max()
df["weight"] = 1.0 + (df["weight"] - w_min) / (w_max - w_min) * 9.0
# df["weight"] *= df["time_decay"]

# filter items that appear fewer than 5 times
item_counts = df["itemid"].value_counts()
valid_items = set(item_counts[item_counts >= 5].index)
df = df[df["itemid"].isin(valid_items)]

# build item vocabulary  (0 = padding)
unique_items = sorted(df["itemid"].unique())
item2idx = {int(item): idx + 1 for idx, item in enumerate(unique_items)}
num_items = len(item2idx) + 1  # +1 for padding idx 0

df["item_idx"] = df["itemid"].map(item2idx)

props1 = pd.read_csv("item_properties_part1.csv")
props2 = pd.read_csv("item_properties_part2.csv")
props = pd.concat([props1, props2], ignore_index=True)
cat_rows = props[props["property"] == "categoryid"][["itemid", "value"]].drop_duplicates("itemid")
cat_rows.columns = ["itemid", "categoryid"]
cat_rows["categoryid"] = pd.to_numeric(cat_rows["categoryid"], errors="coerce")
cat_rows = cat_rows.dropna(subset=["categoryid"])
cat_rows["categoryid"] = cat_rows["categoryid"].astype(int)

cat_rows = cat_rows[cat_rows["itemid"].isin(item2idx)]

unique_cats = sorted(cat_rows["categoryid"].unique())
cat2idx = {cat: idx + 1 for idx, cat in enumerate(unique_cats)}
num_cats = len(unique_cats) + 1

item_to_cat_idx  = {}
for _, row in cat_rows.iterrows():
    if row["itemid"] in item2idx:
        # continue
        item_to_cat_idx[item2idx[row["itemid"]]] = cat2idx.get(int(row["categoryid"]), 0)

df["cat_idx"] = df["item_idx"].map(item_to_cat_idx).fillna(0).astype(int)

print(f"Item vocab: {num_items}")
print(f"Category vocab: {num_cats}")
print(f"Items with category: {len(item_to_cat_idx)} / {num_items - 1}")

# group into per-user sequences (items + weights + inter-event time deltas)
def build_sequence(g):
    items = g["item_idx"].tolist()
    cats = g["cat_idx"].tolist()
    weights = g["weight"].tolist()
    ts = g["ts_seconds"].tolist()
    # time delta in hours between consecutive events (0 for the first)
    deltas = [0.0] + [(ts[i] - ts[i-1]) / 3600.0 for i in range(1, len(ts))]
    return list(zip(items, cats, weights, deltas))

grouped = (
    df.groupby("visitorid")
    .apply(build_sequence)
    .reset_index()
)
grouped.columns = ["visitorid", "sequence"]

# keep sequences of length >= 3
grouped = grouped[grouped["sequence"].apply(len) >= 3]

# cap sequence length to last 50 interactions
MAX_SEQ = 50
grouped["sequence"] = grouped["sequence"].apply(lambda s: s[-MAX_SEQ:])

print(f"Vocab size (incl. pad): {num_items}")
print(f"Sequences: {len(grouped)}")

Item vocab: 90949
Category vocab: 1044
Items with category: 79356 / 90948
Vocab size (incl. pad): 90949
Sequences: 183962


In [17]:
grouped['sequence'][0]

[(55828, 718, 1.0, 0.0),
 (69710, 155, 1.0, 0.047264444496896534),
 (12957, 203, 1.0, 0.043773333297835455)]

In [4]:
item_to_cat_idx[item2idx[6]]
# int(np.int64(265036))

KeyError: 1

In [5]:
len(cat2idx)

1043

265036

In [3]:
class SessionDataset(Dataset):
    """
    Each sample:
      input_items   = item indices for seq[:-1]
      input_weights = event weights (with time decay) for seq[:-1]
      input_deltas  = inter-event time gaps in hours for seq[:-1]
      target        = item index of seq[-1]
      target_weight = event weight of seq[-1]
    """

    def __init__(self, sequences):
        self.input_items = []
        self.input_cats = []
        self.input_weights = []
        self.input_deltas = []
        self.targets = []
        self.target_weights = []

        for seq in sequences:
            items, cats, weights, deltas = zip(*seq)
            self.input_items.append(torch.tensor(items[:-1], dtype=torch.long))
            self.input_cats.append(torch.tensor(cats[:-1], dtype=torch.long))
            self.input_weights.append(torch.tensor(weights[:-1], dtype=torch.float))
            self.input_deltas.append(torch.tensor(deltas[:-1], dtype=torch.float))
            self.targets.append(items[-1])
            self.target_weights.append(weights[-1])

        self.targets = torch.tensor(self.targets, dtype=torch.long)
        self.target_weights = torch.tensor(self.target_weights, dtype=torch.float)

    def __len__(self):
        return len(self.targets)

    def __getitem__(self, idx):
        return (
            self.input_items[idx],
            self.input_cats[idx],
            self.input_weights[idx],
            self.input_deltas[idx],
            self.targets[idx],
            self.target_weights[idx],
        )

def collate_fn(batch):
    items, cats, weights, deltas, targets, target_w = zip(*batch)
    lengths = torch.tensor([len(x) for x in items])
    padded_items = pad_sequence(items, batch_first=True, padding_value=0)
    padded_cats = pad_sequence(cats, batch_first=True, padding_value=0)
    padded_weights = pad_sequence(weights, batch_first=True, padding_value=0.0)
    padded_deltas = pad_sequence(deltas, batch_first=True, padding_value=0.0)
    return padded_items, padded_cats, padded_weights, padded_deltas, lengths, torch.stack(targets), torch.stack(target_w)


# 80/20 train-test split (random)
seqs = grouped["sequence"].tolist()
np.random.seed(42)
np.random.shuffle(seqs)
split = int(0.8 * len(seqs))

train_ds = SessionDataset(seqs[:split])
test_ds = SessionDataset(seqs[split:])

train_loader = DataLoader(train_ds, batch_size=256, shuffle=True, collate_fn=collate_fn)
test_loader = DataLoader(test_ds, batch_size=512, shuffle=False, collate_fn=collate_fn)

In [4]:
class LSTMRec(nn.Module):
    def __init__(self, num_items, embed_dim=64, hidden_dim=128, num_layers=1, dropout=0.2):
        super().__init__()
        self.embedding = nn.Embedding(num_items, embed_dim, padding_idx=0)
        # +1 input dim for the time-delta feature
        self.lstm = nn.LSTM(
            embed_dim + 1, hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim, num_items)

    def forward(self, x, weights, deltas, lengths):
        """
        x:       (B, T)  item indices
        weights: (B, T)  event weights (includes time decay)
        deltas:  (B, T)  inter-event time gap in hours (log-scaled inside)
        lengths: (B,)
        """
        emb = self.embedding(x)                             # (B, T, E)
        emb = emb * weights.unsqueeze(-1)                   # scale by decayed weight

        # log-scale time deltas to compress range (add 1 to avoid log(0))
        time_feat = torch.log1p(deltas).unsqueeze(-1)       # (B, T, 1)

        # concatenate time feature to embedding
        lstm_input = torch.cat([emb, time_feat], dim=-1)    # (B, T, E+1)

        packed = nn.utils.rnn.pack_padded_sequence(
            lstm_input, lengths.cpu(), batch_first=True, enforce_sorted=False
        )
        _, (h_n, _) = self.lstm(packed)
        out = self.dropout(h_n[-1])
        logits = self.fc(out)
        return logits

# class GRURec(nn.Module):
#     def __init__(self):

In [5]:
class Attention(nn.Module):
    """Additive attention over RNN hidden states, conditioned on the last hidden."""
    def __init__(self, hidden_dim):
        super().__init__()
        self.W1 = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.W2 = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.v = nn.Linear(hidden_dim, 1, bias=False)

    def forward(self, hiddens, last_hidden, mask):
        """
        hiddens:     (B, T, H) all RNN outputs
        last_hidden: (B, H)    last valid hidden state
        mask:        (B, T)    True for valid positions
        """
        # (B, T, H) + (B, 1, H) -> (B, T, H)
        energy = self.v(torch.tanh(self.W1(hiddens) + self.W2(last_hidden.unsqueeze(1))))
        energy = energy.squeeze(-1)                  # (B, T)
        energy = energy.masked_fill(~mask, -1e9)
        weights = F.softmax(energy, dim=1)           # (B, T)
        context = (hiddens * weights.unsqueeze(-1)).sum(dim=1)  # (B, H)
        return context, weights

class LSTMRec(nn.Module):
    def __init__(self, num_items, num_cats, item_embed_dim=64, cat_embed_dim=16, hidden_dim=128, num_layers=2, dropout=0.3, embed_dropout=0.2, use_attention=False):
        super().__init__()
        self.item_embedding = nn.Embedding(num_items, item_embed_dim, padding_idx=0)
        self.cat_embedding = nn.Embedding(num_cats, cat_embed_dim, padding_idx=0)
        self.embed_drop = nn.Dropout(embed_dropout)
        self.use_attention = use_attention

        lstm_input_dim = item_embed_dim + cat_embed_dim + 1
        # +1 input dim for the time-delta feature

        self.lstm = nn.LSTM(
            lstm_input_dim, hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout
        )

        if use_attention:
            self.attention = Attention(hidden_dim)
            fc_input_dim = hidden_dim * 2
        else:
            fc_input_dim = hidden_dim

        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(fc_input_dim, item_embed_dim)

        self.output_bias = nn.Parameter(torch.zeros(num_items))

    def forward(self, items, cats, weights, deltas, lengths):
        """
        x:       (B, T)  item indices
        weights: (B, T)  event weights (includes time decay)
        deltas:  (B, T)  inter-event time gap in hours (log-scaled inside)
        lengths: (B,)
        """
        item_emb = self.embed_drop(self.item_embedding(items))                             # (B, T, E)
        cat_emb = self.embed_drop(self.cat_embedding(cats))
        combined = torch.cat((item_emb, cat_emb), dim=-1)
        combined *= weights.unsqueeze(-1)                   # scale by decayed weight

        # log-scale time deltas to compress range (add 1 to avoid log(0))
        time_feat = torch.log1p(deltas).unsqueeze(-1)       # (B, T, 1)
        lstm_input = torch.cat([combined, time_feat], dim=-1)

        packed = nn.utils.rnn.pack_padded_sequence(
            lstm_input, lengths.cpu(), batch_first=True, enforce_sorted=False
        )
        rnn_out, (h_n, _) = self.lstm(packed)
        last_hidden = h_n[-1]
        # print(last_hidden.shape)

        if self.use_attention:
            rnn_out, _ = pad_packed_sequence(rnn_out, batch_first=True)
            B, T = items.size()
            mask = torch.arange(T, device=items.device).unsqueeze(0) < lengths.unsqueeze(1).to(items.device)
            context, _ = self.attention(rnn_out, last_hidden, mask)
            merged = torch.cat([context, last_hidden], dim=-1)
        else:
            merged = last_hidden

        merged = self.dropout(merged)
        proj = self.fc(merged)

        # print(proj.shape)
        # print(self.item_embedding.weight.shape)

        logits = torch.matmul(proj, self.item_embedding.weight.t())
        logits += self.output_bias
        return logits

class GRURec(nn.Module):
    def __init__(self, num_items, num_cats,
                 item_embed_dim=64, cat_embed_dim=16,
                 hidden_dim=128, num_layers=2, dropout=0.3, embed_dropout=0.2, use_attention=False):
        super().__init__()
        self.item_embed_dim = item_embed_dim
        self.item_embedding = nn.Embedding(num_items, item_embed_dim, padding_idx=0)
        self.cat_embedding = nn.Embedding(num_cats, cat_embed_dim, padding_idx=0)
        self.use_attention = use_attention
        self.embed_drop = nn.Dropout(embed_dropout)
        self.hidden_dim = hidden_dim

        input_dim = item_embed_dim + cat_embed_dim + 1

        self.gru = nn.GRU(
            input_dim, hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout,
        )

        if use_attention:
            self.attention = Attention(hidden_dim)
            fc_input_dim = hidden_dim * 2
        else:
            fc_input_dim = hidden_dim

        self.layer_norm = nn.LayerNorm(hidden_dim)
        self.dropout = nn.Dropout(dropout)

        self.fc = nn.Linear(fc_input_dim, item_embed_dim)
        self.output_bias = nn.Parameter(torch.zeros(num_items))

    def forward(self, items, cats, weights, deltas, lengths):
        item_emb = self.embed_drop(self.item_embedding(items))
        cat_emb = self.embed_drop(self.cat_embedding(cats))
        combined = torch.cat([item_emb, cat_emb], dim=-1)
        combined = combined * weights.unsqueeze(-1)

        time_feat = torch.log1p(deltas).unsqueeze(-1)
        raw_input = torch.cat([combined, time_feat], dim=-1)  # (B, T, input_dim)

        packed = pack_padded_sequence(raw_input, lengths.cpu(), batch_first=True, enforce_sorted=False)
        rnn_out, h_n = self.gru(packed)
        rnn_out, _ = nn.utils.rnn.pad_packed_sequence(rnn_out, batch_first=True)  # (B, T, H)

        # residual connection + layer norm
        # rnn_out and projected may differ in T due to padding, slice to match

        rnn_out = self.layer_norm(rnn_out)

        last_hidden = h_n[-1]

        if self.use_attention:
            T_out = rnn_out.size(1)
            mask = torch.arange(T_out, device=items.device).unsqueeze(0) < lengths.unsqueeze(1).to(items.device)
            context, _ = self.attention(rnn_out, last_hidden, mask)
            merged = torch.cat([context, last_hidden], dim=-1)
        else:
            merged = last_hidden

        merged = self.dropout(merged)
        proj = self.fc(merged)

        logits = F.linear(proj, self.item_embedding.weight, self.output_bias)
        return logits

class SASRec(nn.Module):
    """
    Self-Attentive Sequential Recommendation (Kang & McAuley, 2018).
    Causal transformer encoder over item sequences.
    Uses the last position's output as the sequence representation.
    """

    def __init__(self, num_items, num_cats,
                 item_embed_dim=64, cat_embed_dim=16,
                 hidden_dim=128, n_heads=4, n_layers=2,
                 dropout=0.3, embed_dropout=0.2, max_len=50):
        super().__init__()
        self.item_embed_dim = item_embed_dim
        self.hidden_dim = hidden_dim
        self.max_len = max_len

        self.item_embedding = nn.Embedding(num_items, item_embed_dim, padding_idx=0)
        self.cat_embedding = nn.Embedding(num_cats, cat_embed_dim, padding_idx=0)
        self.embed_drop = nn.Dropout(embed_dropout)

        input_dim = item_embed_dim + cat_embed_dim + 1
        self.input_proj = nn.Linear(input_dim, hidden_dim)

        self.pos_embedding = nn.Embedding(max_len, hidden_dim)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=hidden_dim,
            nhead=n_heads,
            dim_feedforward=hidden_dim * 4,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.layer_norm = nn.LayerNorm(hidden_dim)

        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim, item_embed_dim)
        self.output_bias = nn.Parameter(torch.zeros(num_items))

        print(f"  [SASRec] {n_layers} layers, {n_heads} heads, hidden={hidden_dim}")

    def forward(self, items, cats, weights, deltas, lengths):
        B, T = items.size()

        item_emb = self.embed_drop(self.item_embedding(items))
        cat_emb = self.embed_drop(self.cat_embedding(cats))
        combined = torch.cat([item_emb, cat_emb], dim=-1)
        combined = combined * weights.unsqueeze(-1)
        time_feat = torch.log1p(deltas).unsqueeze(-1)
        features = torch.cat([combined, time_feat], dim=-1)

        hidden = self.input_proj(features)
        positions = torch.arange(T, device=items.device).unsqueeze(0).expand(B, -1)
        hidden = hidden + self.pos_embedding(positions)
        hidden = self.layer_norm(hidden)

        causal_mask = torch.triu(
            torch.ones(T, T, device=items.device, dtype=torch.bool), diagonal=1
        )
        padding_mask = torch.arange(T, device=items.device).unsqueeze(0) >= lengths.unsqueeze(1).to(items.device)

        out = self.transformer(
            hidden,
            mask=causal_mask,
            src_key_padding_mask=padding_mask,
        )

        last_idx = (lengths - 1).long().to(items.device)
        last_hidden = out[torch.arange(B, device=items.device), last_idx]

        proj = self.fc(self.dropout(last_hidden))

        logits = torch.matmul(proj, self.item_embedding.weight.t())
        logits = logits + self.output_bias
        return logits


def compute_loss(logits, targets, target_weights):
    """Weighted cross-entropy loss."""
    per_sample = F.cross_entropy(logits, targets, reduction="none")
    return (per_sample * target_weights).mean()

class BPRLoss(nn.Module):
    def __init__(self, num_items, n_negatives=5):
        super().__init__()
        self.num_items = num_items
        self.n_neg = n_negatives

    def forward(self, logits, targets, target_weights):
        """
        logits:         (B, num_items)
        targets:        (B,) positive item indices
        target_weights: (B,) sample importance weights
        """
        pos_scores = logits.gather(1, targets.unsqueeze(1))  # (B, 1)

        # sample random negatives
        neg_idx = torch.randint(1, self.num_items, (logits.size(0), self.n_neg), device=logits.device)
        neg_scores = logits.gather(1, neg_idx)  # (B, n_neg)

        # BPR: -log(sigmoid(pos - neg)), averaged over negatives
        diff = pos_scores - neg_scores  # (B, n_neg)
        bpr = -F.logsigmoid(diff).mean(dim=1)  # (B,)

        # weight by event importance
        loss = (bpr * target_weights).mean()
        return loss

In [6]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# model = LSTMRec(num_items, num_cats).to(device)
# optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
# criterion = nn.CrossEntropyLoss(reduction="none")  # per-sample so we can weight
# criterion = BPRLoss(num_items, n_negatives=10)  # per-sample so we can weight

EPOCHS = 10

def train_model(model, name):
    print(f"\n{'='*50}")
    print(f"  Training {name}")
    print(f"{'='*50}")

    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-5)
    criterion = nn.CrossEntropyLoss(reduction="none")  # per-sample so we can weight

    # for epoch in range(1, EPOCHS + 1):
    #     model.train()
    #     total_loss = 0
    #     n_samples = 0
    # # debug: verify batch structure on first iteration
    # sample_batch = next(iter(train_loader))
    # print(f"  Batch has {len(sample_batch)} elements")
    # for i, t in enumerate(sample_batch):
    #     print(f"    [{i}] shape={t.shape} dtype={t.dtype}")

    for epoch in range(1, EPOCHS + 1):
        model.train()
        total_loss = 0
        n_samples = 0
        for batch in train_loader:
            p_items = batch[0].to(device)
            p_cats = batch[1].to(device)
            p_weights = batch[2].to(device)
            p_deltas = batch[3].to(device)
            lengths = batch[4].cpu()
            targets = batch[5].to(device)
            target_w = batch[6].to(device)

            logits = model(p_items, p_cats, p_weights, p_deltas, lengths)
            loss = compute_loss(logits, targets, target_w)

            optimizer.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            total_loss += loss.item() * targets.size(0)
            n_samples += targets.size(0)

        scheduler.step()
        lr = optimizer.param_groups[0]["lr"]
        print(f"  Epoch {epoch}/{EPOCHS}  loss={total_loss/n_samples:.4f}  lr={lr:.6f}")

    return model

lstm_model = train_model(LSTMRec(num_items, num_cats), "LSTM + Attention")
gru_model = train_model(GRURec(num_items, num_cats), "GRU + Attention")
sas_model = train_model(SASRec(num_items, num_cats), "SAS + Attention")

# for epoch in range(1, EPOCHS + 1):
#     model.train()
#     total_loss = 0
#     n_samples = 0
#     for batch in train_loader:
#         padded_items = batch[0].to(device)
#         padded_cats = batch[1].to(device)
#         padded_weights = batch[2].to(device)
#         padded_deltas = batch[3].to(device)
#         lengths = batch[4].to(device)
#         targets = batch[5].to(device)
#         target_w = batch[6].to(device)
#
#         logits = model(padded_items, padded_cats, padded_weights, padded_deltas, lengths)
#         per_sample_loss = criterion(logits, targets, target_w)
#
#         # weight the loss: predicting a purchased item correctly
#         # matters more than predicting a viewed item
#         weighted_loss = (per_sample_loss * target_w).mean()
#
#         optimizer.zero_grad()
#         weighted_loss.backward()
#         optimizer.step()
#         total_loss += weighted_loss.item() * targets.size(0)
#
#     avg_loss = total_loss / len(train_ds)
#     print(f"Epoch {epoch}/{EPOCHS}  train_loss={avg_loss:.4f}")


  Training LSTM + Attention
  Epoch 1/10  loss=11.6352  lr=0.000976
  Epoch 2/10  loss=9.3576  lr=0.000905
  Epoch 3/10  loss=8.8663  lr=0.000796
  Epoch 4/10  loss=8.5455  lr=0.000658
  Epoch 5/10  loss=8.2872  lr=0.000505
  Epoch 6/10  loss=8.0582  lr=0.000352
  Epoch 7/10  loss=7.8813  lr=0.000214
  Epoch 8/10  loss=7.7196  lr=0.000105
  Epoch 9/10  loss=7.6454  lr=0.000034
  Epoch 10/10  loss=7.5915  lr=0.000010

  Training GRU + Attention
  Epoch 1/10  loss=10.8594  lr=0.000976
  Epoch 2/10  loss=9.3382  lr=0.000905
  Epoch 3/10  loss=8.9672  lr=0.000796
  Epoch 4/10  loss=8.6948  lr=0.000658
  Epoch 5/10  loss=8.4333  lr=0.000505
  Epoch 6/10  loss=8.2009  lr=0.000352
  Epoch 7/10  loss=8.0289  lr=0.000214
  Epoch 8/10  loss=7.8993  lr=0.000105
  Epoch 9/10  loss=7.8178  lr=0.000034
  Epoch 10/10  loss=7.7619  lr=0.000010
  [SASRec] 2 layers, 4 heads, hidden=128

  Training SAS + Attention


C:\Users\User\AppData\Local\Temp\ipykernel_16980\974412569.py:195: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)


  Epoch 1/10  loss=12.1536  lr=0.000976
  Epoch 2/10  loss=9.7176  lr=0.000905
  Epoch 3/10  loss=9.3550  lr=0.000796
  Epoch 4/10  loss=9.0800  lr=0.000658
  Epoch 5/10  loss=8.8380  lr=0.000505
  Epoch 6/10  loss=8.5864  lr=0.000352
  Epoch 7/10  loss=8.3645  lr=0.000214
  Epoch 8/10  loss=8.1755  lr=0.000105
  Epoch 9/10  loss=8.0680  lr=0.000034
  Epoch 10/10  loss=7.9812  lr=0.000010


In [7]:
def evaluate(model, loader, ks=(5, 10, 20)):
    model.eval()
    hits = {k: 0 for k in ks}
    mrr = {k: 0.0 for k in ks}
    ndcg = {k: 0.0 for k in ks}
    total = 0

    with torch.no_grad():
        for padded_items, padded_cats, padded_weights, padded_deltas, lengths, targets, _ in loader:
            padded_items = padded_items.to(device)
            padded_cats = padded_cats.to(device)
            padded_weights = padded_weights.to(device)
            padded_deltas = padded_deltas.to(device)
            targets = targets.to(device)

            logits = model(padded_items, padded_cats, padded_weights, padded_deltas, lengths)

            for k in ks:
                topk = logits.topk(k, dim=1).indices
                match = (topk == targets.unsqueeze(1))  # bool (B, k)

                has_hit = match.any(dim=1)  # bool (B,)
                # position of hit (1-indexed), 0 if no hit
                hit_rank = match.float().argmax(dim=1) + 1  # (B,)
                hit_rank[~has_hit] = 0

                # HR@K (= Recall@K when 1 relevant item)
                hits[k] += has_hit.sum().item()

                # MRR@K
                rr = torch.zeros_like(hit_rank, dtype=torch.float)
                rr[has_hit] = 1.0 / hit_rank[has_hit].float()
                mrr[k] += rr.sum().item()

                # NDCG@K: 1/log2(rank+1) if hit, else 0
                dcg = torch.zeros_like(hit_rank, dtype=torch.float)
                dcg[has_hit] = 1.0 / torch.log2(hit_rank[has_hit].float() + 1)
                ndcg[k] += dcg.sum().item()

            total += targets.size(0)

    print("\n── Test Metrics ──")
    for k in ks:
        hr = hits[k] / total
        print(f"  @{k:>2} HR= {hr:.4f} \tPrecision={hr/k:.4f} \tRecall={hr:.4f} \tMAP={mrr[k]/total:.4f} \tMRR@{k:>2} = {mrr[k]/total:.4f} \tNDCG={ndcg[k]/total:.4f}")


evaluate(lstm_model, test_loader)
evaluate(gru_model, test_loader)
evaluate(sas_model, test_loader)


── Test Metrics ──
  @ 5 HR= 0.3384 	Precision=0.0677 	Recall=0.3384 	MAP=0.2977 	MRR@ 5 = 0.2977 	NDCG=0.3080
  @10 HR= 0.3550 	Precision=0.0355 	Recall=0.3550 	MAP=0.3000 	MRR@10 = 0.3000 	NDCG=0.3134
  @20 HR= 0.3691 	Precision=0.0185 	Recall=0.3691 	MAP=0.3009 	MRR@20 = 0.3009 	NDCG=0.3170

── Test Metrics ──
  @ 5 HR= 0.3351 	Precision=0.0670 	Recall=0.3351 	MAP=0.2951 	MRR@ 5 = 0.2951 	NDCG=0.3053
  @10 HR= 0.3500 	Precision=0.0350 	Recall=0.3500 	MAP=0.2972 	MRR@10 = 0.2972 	NDCG=0.3101
  @20 HR= 0.3648 	Precision=0.0182 	Recall=0.3648 	MAP=0.2982 	MRR@20 = 0.2982 	NDCG=0.3139

── Test Metrics ──
  @ 5 HR= 0.3074 	Precision=0.0615 	Recall=0.3074 	MAP=0.2689 	MRR@ 5 = 0.2689 	NDCG=0.2787
  @10 HR= 0.3236 	Precision=0.0324 	Recall=0.3236 	MAP=0.2711 	MRR@10 = 0.2711 	NDCG=0.2839
  @20 HR= 0.3382 	Precision=0.0169 	Recall=0.3382 	MAP=0.2721 	MRR@20 = 0.2721 	NDCG=0.2876


In [10]:
import pickle

artifacts = {
    "item2idx": item2idx,
    "idx2item": {idx: item for item, idx in item2idx.items()},
    "cat2idx": cat2idx,
    "item_to_cat_idx": item_to_cat_idx,
    "num_items": num_items,
    "num_cats": num_cats,
    "event_weights": EVENT_WEIGHTS,
    "max_seq": MAX_SEQ
}

with open("./artifacts/artifacts.pkl", "wb") as f:
    pickle.dump(artifacts, f)
print("Saved artifacts")

torch.save(lstm_model.state_dict(), "artifacts/lstm_model.pt")
torch.save(gru_model.state_dict(), "artifacts/gru_model.pt")
torch.save(sas_model.state_dict(), "artifacts/sas_model.pt")

Saved artifacts


In [27]:
"import itertools
for padded_items, padded_cats, padded_weights, padded_deltas, lengths, targets, *_ in itertools.islice(test_loader, 5):
    padded_items = padded_items.to(device)
    padded_cats = padded_cats.to(device)
    padded_weights = padded_weights.to(device)
    padded_deltas = padded_deltas.to(device)
    targets = targets.to(device)
    print(lstm_model(padded_items, padded_cats, padded_weights, padded_deltas, lengths).topk(5, dim=1).indices)

tensor([[47887, 75929, 45492, 44513, 37658],
        [17934, 27702,  2258, 30809, 16191],
        [54102, 15050, 53101, 83847,  2122],
        ...,
        [49283, 30105, 19976, 60934, 25875],
        [68337, 57244,  8551, 21972, 40359],
        [ 2873, 79724, 11087, 50250, 58509]], device='cuda:0')
tensor([[22691, 43148,  2430, 63362, 41547],
        [24268, 62503, 54754, 43835, 85949],
        [65191,  3664, 13843,  8883, 82476],
        ...,
        [40875, 65362, 74117, 74721, 37550],
        [69900,  5071, 56646, 51363, 23782],
        [53467, 44536, 66272, 70626,  5858]], device='cuda:0')
tensor([[80295, 89218, 48907, 74170, 70784],
        [50234, 70275, 66411, 36578, 10654],
        [23457, 77203, 54389, 74888, 83525],
        ...,
        [84517, 59604,  6108, 87638, 51212],
        [76922, 40896,  8765, 81127, 55263],
        [34891, 43835,  3809, 50263, 69389]], device='cuda:0')
tensor([[47944, 12097, 67461, 38949,  7898],
        [89959, 21128, 42661,  5209,  1994],
       

In [ ]:
props = pd.read_csv("item_properties_part1.csv")
cat_rows = props[props["property"] == "categoryid"][["itemid", "value"]].drop_duplicates("itemid")
cat_rows.columns = ["itemid", "categoryid"]
cat_rows["categoryid"] = pd.to_numeric(cat_rows["categoryid"], errors="coerce")
cat_rows = cat_rows.dropna(subset=["categoryid"])
cat_rows["categoryid"] = cat_rows["categoryid"].astype(int)

# map item_idx → categoryid
item_idx_to_cat = {}
for _, row in cat_rows.iterrows():
    if row["itemid"] in item2idx:
        item_idx_to_cat[item2idx[row["itemid"]]] = row["categoryid"]

print(f"\nItems with category info: {len(item_idx_to_cat)} / {num_items - 1}")

# --- 6b. Build per-user category affinity scores ---
def build_user_category_affinity(sequences):
    """
    For each user sequence, accumulate category weights.
    Returns dict: seq_index -> {categoryid: total_weight}
    """
    affinities = {}
    for i, seq in enumerate(sequences):
        cat_scores = {}
        for item_idx, weight, delta in seq:
            cat = item_idx_to_cat.get(item_idx)
            if cat is not None:
                cat_scores[cat] = cat_scores.get(cat, 0.0) + weight
        affinities[i] = cat_scores
    return affinities


# --- 6c. Boosted inference ---
def recommend_with_category_boost(model, dataset, sequences, top_k=20, boost=1.5):
    """
    Get model scores, then multiply logits for items in the user's
    preferred categories by a boost factor.

    boost: multiplier for items in top-affinity categories.
    """
    model.eval()
    affinities = build_user_category_affinity(sequences)

    # precompute item_idx → category tensor for fast boosting
    cat_tensor = torch.zeros(num_items, dtype=torch.long)
    for idx, cat in item_idx_to_cat.items():
        cat_tensor[idx] = cat

    loader = DataLoader(dataset, batch_size=512, shuffle=False, collate_fn=collate_fn)
    all_recs = []
    sample_idx = 0

    with torch.no_grad():
        for padded_items, padded_weights, padded_deltas, lengths, targets, _ in loader:
            padded_items = padded_items.to(device)
            padded_weights = padded_weights.to(device)
            padded_deltas = padded_deltas.to(device)
            logits = model(padded_items, padded_weights, padded_deltas, lengths)  # (B, num_items)

            # apply category boost per user in batch
            for i in range(logits.size(0)):
                user_aff = affinities.get(sample_idx + i, {})
                if user_aff:
                    # pick top-3 favourite categories for this user
                    top_cats = sorted(user_aff, key=user_aff.get, reverse=True)[:3]
                    top_cats_set = set(top_cats)
                    # build a mask of items belonging to favourite categories
                    mask = torch.zeros(num_items, device=device)
                    for idx in range(num_items):
                        if cat_tensor[idx].item() in top_cats_set:
                            mask[idx] = 1.0
                    # boost: multiply logits by boost where mask=1, keep rest at 1.0
                    multiplier = 1.0 + mask * (boost - 1.0)
                    logits[i] = logits[i] * multiplier

            topk_items = logits.topk(top_k, dim=1).indices  # (B, top_k)
            all_recs.append(topk_items.cuda())
            sample_idx += logits.size(0)

    return torch.cat(all_recs, dim=0)


# --- 6d. Compare base vs boosted ---
def evaluate_recommendations(recs, dataset, ks=(5, 10, 20)):
    targets = dataset.targets
    results = {}
    for k in ks:
        topk = recs[:, :k]
        match = (topk == targets.unsqueeze(1)).float()
        hr = match.sum().item() / len(targets)
        ranks = match * torch.arange(1, k + 1).float()
        ranks[ranks == 0] = float("inf")
        m = (1.0 / ranks.min(dim=1).values).sum().item() / len(targets)
        results[k] = (hr, m)
    return results

In [11]:
from collections import defaultdict

def mine_sequential_rules(sequences, min_support_count=5):
    """
    Mine pairwise sequential rules (A → 😎 from training sequences.
    A transition is any (seq[i], seq[i+1]) consecutive pair.

    Returns:
        confidence: dict of {(A, B): float}
        support:    dict of {(A, B): float}
    """
    pair_counts = defaultdict(int)
    item_counts = defaultdict(int)  # count of item as antecedent
    total_transitions = 0

    for seq in sequences:
        items = [item for item, _, _ in seq]
        for i in range(len(items) - 1):
            a, b = items[i], items[i + 1]
            pair_counts[(a, b)] += 1
            item_counts[a] += 1
            total_transitions += 1

    # filter by minimum support count to reduce noise
    confidence = {}
    support = {}
    for (a, b), count in pair_counts.items():
        if count >= min_support_count:
            confidence[(a, b)] = count / item_counts[a]
            support[(a, b)] = count / total_transitions

    print(f"\nMined {len(confidence)} association rules "
          f"(min_support_count={min_support_count})")

    # show top rules by confidence
    top_rules = sorted(confidence.items(), key=lambda x: x[1], reverse=True)[:10]
    print("Top 10 rules by confidence:")
    for (a, b), conf in top_rules:
        sup = support[(a, b)]
        print(f"  item {a} → item {b}:  conf={conf:.3f}  sup={sup:.6f}")

    return confidence, support


# build rules from training sequences only (no data leakage)
train_seqs = seqs[:split]
rule_confidence, rule_support = mine_sequential_rules(train_seqs, min_support_count=5)

ALPHA = 1.0
BETA = 1000.0  # support values are very small, scale to be comparable

rule_lookup = defaultdict(dict)  # {antecedent_item: {consequent_item: score}}
for (a, b), conf in rule_confidence.items():
    sup = rule_support[(a, b)]
    rule_lookup[a][b] = ALPHA * conf + BETA * sup


Mined 19647 association rules (min_support_count=5)
Top 10 rules by confidence:
  item 54059 → item 54059:  conf=1.000  sup=0.000008
  item 66324 → item 66324:  conf=1.000  sup=0.000007
  item 66136 → item 66136:  conf=1.000  sup=0.000011
  item 79722 → item 79722:  conf=1.000  sup=0.000014
  item 45316 → item 45316:  conf=1.000  sup=0.000011
  item 60749 → item 60749:  conf=1.000  sup=0.000007
  item 13771 → item 13771:  conf=1.000  sup=0.000007
  item 27655 → item 27655:  conf=1.000  sup=0.000008
  item 38612 → item 38612:  conf=1.000  sup=0.000018
  item 402 → item 402:  conf=1.000  sup=0.000011


In [12]:
def recommend_with_all_boosts(
    model, dataset, sequences,
    top_k=20,
    category_boost=1.5,
    rule_boost_weight=0.3,
    n_recent=3,
):
    """
    Combined reranking:
    1. Get base model logits
    2. Apply category affinity boost (multiplicative)
    3. Apply association rule boost (additive on logits)
       based on the last n_recent items in the sequence

    rule_boost_weight: how much the rule score adds to logits
    n_recent: how many recent items to look back for rule matching
    """
    model.eval()
    affinities = build_user_category_affinity(sequences)

    cat_tensor = torch.zeros(num_items, dtype=torch.long)
    for idx, cat in item_idx_to_cat.items():
        cat_tensor[idx] = cat

    loader = DataLoader(dataset, batch_size=512, shuffle=False, collate_fn=collate_fn)
    all_recs = []
    sample_idx = 0

    with torch.no_grad():
        for padded_items, padded_weights, padded_deltas, lengths, targets, _ in loader:
            padded_items = padded_items.to(device)
            padded_weights = padded_weights.to(device)
            padded_deltas = padded_deltas.to(device)
            logits = model(padded_items, padded_weights, padded_deltas, lengths)

            for i in range(logits.size(0)):
                # --- category boost (multiplicative) ---
                user_aff = affinities.get(sample_idx + i, {})
                if user_aff:
                    top_cats = sorted(user_aff, key=user_aff.get, reverse=True)[:3]
                    top_cats_set = set(top_cats)
                    mask = torch.zeros(num_items, device=device)
                    for idx in range(num_items):
                        if cat_tensor[idx].item() in top_cats_set:
                            mask[idx] = 1.0
                    multiplier = 1.0 + mask * (category_boost - 1.0)
                    logits[i] = logits[i] * multiplier

                # --- association rule boost (additive) ---
                seq = sequences[sample_idx + i]
                recent_items = [item for item, _, _ in seq[-n_recent:]]
                rule_scores = torch.zeros(num_items, device=device)
                for antecedent in recent_items:
                    if antecedent in rule_lookup:
                        for consequent, score in rule_lookup[antecedent].items():
                            if consequent < num_items:
                                rule_scores[consequent] += score
                # normalize rule scores to be on similar scale as logits
                if rule_scores.max() > 0:
                    rule_scores = rule_scores / rule_scores.max()
                    logits[i] = logits[i] + rule_boost_weight * rule_scores

            topk_items = logits.topk(top_k, dim=1).indices
            all_recs.append(topk_items.cpu())
            sample_idx += logits.size(0)

    return torch.cat(all_recs, dim=0)

In [ ]:
print("\n── Base Model (no boost) ──")
base_recs = recommend_with_category_boost(model, test_ds, seqs[split:], boost=1.0)
base_res = evaluate_recommendations(base_recs, test_ds)
# for k, (hr, m) in base_res.items():
#     print(f"  HR@{k:>2} = {hr:.4f}   MRR@{k:>2} = {m:.4f}")

# print("\n── With Category Boost (1.5x) ──")
# boosted_recs = recommend_with_category_boost(model, test_ds, seqs[split:], boost=1.5)
# boosted_res = evaluate_recommendations(boosted_recs, test_ds)
# for k, (hr, m) in boosted_res.items():
#     print(f"  HR@{k:>2} = {hr:.4f}   MRR@{k:>2} = {m:.4f}")
#
# print("\n── With Category Boost + Association Rules ──")
# combined_recs = recommend_with_all_boosts(model, test_ds, seqs[split:],
#                                           category_boost=1.5, rule_boost_weight=0.3)
# combined_res = evaluate_recommendations(combined_recs, test_ds)
# for k, (hr, m) in combined_res.items():
#     print(f"  HR@{k:>2} = {hr:.4f}   MRR@{k:>2} = {m:.4f}")


── Base Model (no boost) ──


In [ ]:
# torch.cuda.init()
torch.cuda.is_available()

In [31]:
import os
print(os.environ.get('LD_LIBRARY_PATH'))

None
